In [1]:
!pip install fastapi uvicorn joblib pandas numpy "scikit-learn==1.7.2"


   ---------- ----------------------------- 1/4 [uvicorn]
   ---------- ----------------------------- 1/4 [uvicorn]
   -------------------- ------------------- 2/4 [starlette]
   -------------------- ------------------- 2/4 [starlette]
   ------------------------------ --------- 3/4 [fastapi]
   ------------------------------ --------- 3/4 [fastapi]
   ------------------------------ --------- 3/4 [fastapi]
   ---------------------------------------- 4/4 [fastapi]



In [2]:
import joblib
import pandas as pd
import numpy as np
import sklearn

print("Scikit-learn version:", sklearn.__version__)

Scikit-learn version: 1.7.2


In [3]:
MODEL_PATH = "faers_serious_event_model.pkl"

try:
    model = joblib.load(MODEL_PATH)
    print("Model loaded successfully")
except Exception as e:
    print("Error loading model:", e)
    model = None

Model loaded successfully


In [4]:
print(type(model))

<class 'sklearn.pipeline.Pipeline'>


In [5]:
print(model)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['patient_age_years',
                                                   'patient_weight_kg',
                                                   'patient_sex', 'num_drugs',
                                                   'num_reactions',
                                                   'reporttype',
                                                   'qualification',
                                                   'valid_weight',
                                                   'polypharmacy_flag',
 

In [6]:
feature_names = model.feature_names_in_

print("Number of features:", len(feature_names))

for i, feature in enumerate(feature_names, start=1):
    print(i, feature)

Number of features: 20
1 patient_age_years
2 patient_weight_kg
3 patient_sex
4 num_drugs
5 num_reactions
6 reporttype
7 qualification
8 primarysourcecountry
9 occurcountry
10 reportercountry
11 age_group
12 valid_weight
13 polypharmacy_flag
14 reaction_diversity
15 country_match_flag
16 reporting_delay_days
17 suspect_drug_count
18 unique_active_substances
19 unique_indications
20 drug_route_count


In [7]:
print("Model classes:", model.classes_)

Model classes: [0 1]


In [8]:
sample_data = pd.DataFrame({
    "patient_age_years": [55],
    "patient_weight_kg": [70],
    "patient_sex": [1],
    "num_drugs": [6],
    "num_reactions": [3],
    "reporttype": [1],
    "qualification": [1],
    "primarysourcecountry": ["US"],
    "occurcountry": ["US"],
    "reportercountry": ["US"],
    "age_group": ["51-65"],
    "valid_weight": [1],
    "polypharmacy_flag": [1],
    "reaction_diversity": [3],
    "country_match_flag": [1],
    "reporting_delay_days": [5],
    "suspect_drug_count": [2],
    "unique_active_substances": [4],
    "unique_indications": [3],
    "drug_route_count": [2]
})

In [9]:
sample_data

,patient_age_years,patient_weight_kg,patient_sex,num_drugs,num_reactions,reporttype,qualification,primarysourcecountry,occurcountry,reportercountry,age_group,valid_weight,polypharmacy_flag,reaction_diversity,country_match_flag,reporting_delay_days,suspect_drug_count,unique_active_substances,unique_indications,drug_route_count
0,55,70,1,6,3,1,1,US,US,US,51-65,1,1,3,1,5,2,4,3,2


In [10]:
prediction = model.predict(sample_data)[0]

serious_probability = model.predict_proba(sample_data)[0, 1]

if prediction == 1:
    predicted_class = "Serious"
else:
    predicted_class = "Non-Serious"

print("Predicted Seriousness:", predicted_class)
print(
    "Probability of Serious Event:",
    round(float(serious_probability), 4)
)

Predicted Seriousness: Non-Serious
Probability of Serious Event: 0.0


In [11]:
%%writefile app.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import joblib


# --------------------------------------------------
# Create FastAPI application
# --------------------------------------------------

app = FastAPI(
    title="FAERS Serious Adverse Event Risk Prediction API",
    description=(
        "Predicts whether an adverse-event report is "
        "Serious or Non-Serious."
    ),
    version="1.0.0"
)


# --------------------------------------------------
# Load trained ML pipeline
# --------------------------------------------------

MODEL_PATH = "faers_serious_event_model.pkl"

try:
    model = joblib.load(MODEL_PATH)
    print("Model loaded successfully")
except Exception as e:
    print("Error loading model:", e)
    model = None


# --------------------------------------------------
# Define API input schema
# --------------------------------------------------

class ClassificationInput(BaseModel):

    patient_age_years: float
    patient_weight_kg: float
    patient_sex: int

    num_drugs: int
    num_reactions: int

    reporttype: int
    qualification: int

    primarysourcecountry: str
    occurcountry: str
    reportercountry: str

    age_group: str

    valid_weight: int
    polypharmacy_flag: int
    reaction_diversity: int
    country_match_flag: int

    reporting_delay_days: float

    suspect_drug_count: int
    unique_active_substances: int
    unique_indications: int
    drug_route_count: int


# --------------------------------------------------
# Home endpoint
# --------------------------------------------------

@app.get("/")
def home():

    return {
        "message":
        "FAERS Serious Adverse Event Risk Prediction API is running",
        "status": "success"
    }


# --------------------------------------------------
# Health endpoint
# --------------------------------------------------

@app.get("/health")
def health():

    if model is None:

        return {
            "status": "unhealthy",
            "model_loaded": False
        }

    return {
        "status": "healthy",
        "model_loaded": True
    }


# --------------------------------------------------
# Prediction endpoint
# --------------------------------------------------

@app.post("/predict")
def predict(data: ClassificationInput):

    if model is None:

        raise HTTPException(
            status_code=500,
            detail="Model is not loaded"
        )

    try:

        input_data = pd.DataFrame([data.model_dump()])

        prediction = int(
            model.predict(input_data)[0]
        )

        probabilities = model.predict_proba(
            input_data
        )[0]

        serious_probability = float(
            probabilities[1]
        )

        non_serious_probability = float(
            probabilities[0]
        )

        predicted_class = (
            "Serious"
            if prediction == 1
            else "Non-Serious"
        )

        return {
            "prediction": predicted_class,
            "predicted_class": prediction,
            "serious_probability":
                round(serious_probability, 4),
            "non_serious_probability":
                round(non_serious_probability, 4)
        }

    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

Writing app.py


In [12]:
with open("app.py", "r") as file:
    print(file.read())


from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import joblib


# --------------------------------------------------
# Create FastAPI application
# --------------------------------------------------

app = FastAPI(
    title="FAERS Serious Adverse Event Risk Prediction API",
    description=(
        "Predicts whether an adverse-event report is "
        "Serious or Non-Serious."
    ),
    version="1.0.0"
)


# --------------------------------------------------
# Load trained ML pipeline
# --------------------------------------------------

MODEL_PATH = "faers_serious_event_model.pkl"

try:
    model = joblib.load(MODEL_PATH)
    print("Model loaded successfully")
except Exception as e:
    print("Error loading model:", e)
    model = None


# --------------------------------------------------
# Define API input schema
# --------------------------------------------------

class ClassificationInput(BaseModel):

    patient_age_years: float

In [13]:
%%writefile requirements.txt

fastapi
uvicorn
scikit-learn==1.7.2
joblib
numpy
pandas

Writing requirements.txt


In [14]:
with open("requirements.txt", "r") as file:
    print(file.read())


fastapi
uvicorn
scikit-learn==1.7.2
joblib
numpy
pandas



In [15]:
import os

print(os.listdir())

['.ipynb_checkpoints', 'app.py', 'EDA.ipynb', 'FAERS_FastAPI_Deployment.ipynb', 'faers_serious_event_model.pkl', 'Reg or Class Code Blueprint.ipynb', 'requirements.txt']


In [16]:
import app

print("FastAPI app imported successfully")
print("Model loaded:", app.model is not None)

Model loaded successfully
FastAPI app imported successfully
Model loaded: True


In [17]:
test_input = app.ClassificationInput(
    patient_age_years=55,
    patient_weight_kg=70,
    patient_sex=1,
    num_drugs=6,
    num_reactions=3,
    reporttype=1,
    qualification=1,
    primarysourcecountry="US",
    occurcountry="US",
    reportercountry="US",
    age_group="51-65",
    valid_weight=1,
    polypharmacy_flag=1,
    reaction_diversity=3,
    country_match_flag=1,
    reporting_delay_days=5,
    suspect_drug_count=2,
    unique_active_substances=4,
    unique_indications=3,
    drug_route_count=2
)

result = app.predict(test_input)

result

{'prediction': 'Non-Serious',
 'predicted_class': 0,
 'serious_probability': 0.0,
 'non_serious_probability': 1.0}

In [19]:
import os
print(os.getcwd())

C:\Users\Sushma Shekar\sushma python classes\ML Project\notebook


### Create README File

The README file explains the purpose of the project,
the ML model, API endpoints, and deployment.

In [20]:
%%writefile README.md

# FAERS Serious Adverse Event Risk Prediction API

## Project Overview

This project uses machine learning to predict whether a newly
received adverse-event report is likely to be Serious or Non-Serious.

The model is deployed as a REST API using FastAPI.

## ML Problem

Binary Classification

Target:
- 0 = Non-Serious
- 1 = Serious

## Model

Logistic Regression with preprocessing implemented using a
Scikit-learn Pipeline.

## API Framework

FastAPI

## API Endpoints

### GET /
Checks whether the API is running.

### GET /health
Checks whether the trained machine learning model has loaded
successfully.

### POST /predict
Accepts adverse-event information and returns the predicted
seriousness and probability.

## Technologies Used

- Python
- Pandas
- NumPy
- Scikit-learn
- Joblib
- FastAPI
- Uvicorn

## Deployment

The API is deployed using Render.

Writing README.md


In [21]:
import os

os.listdir()

['.ipynb_checkpoints',
 'app.py',
 'EDA.ipynb',
 'FAERS_FastAPI_Deployment.ipynb',
 'faers_serious_event_model.pkl',
 'README.md',
 'Reg or Class Code Blueprint.ipynb',
 'requirements.txt',
 '__pycache__']

In [22]:
import os

size_mb = os.path.getsize(
    "faers_serious_event_model.pkl"
) / (1024 * 1024)

print("Model size:", round(size_mb, 2), "MB")

Model size: 0.01 MB


In [23]:
with open("requirements.txt", "r") as file:
    print(file.read())


fastapi
uvicorn
scikit-learn==1.7.2
joblib
numpy
pandas

